# Overlapping Data Transfers and Computation

The performance of our halo exchanges is already improved, but they still represent a performance overhead.
Depending on parameters, hardware and application, this overhead can anything between negligible and dominating.
One popular optimization is **overlapping** halo exchanges with useful work.

The core idea is to
* compute the data to be communicated first, i.e. compute the first and last row in our example
* launch halo transfers as soon as possible and asynchronously, and
* hide the transfers behind the bulk computation.

Computing the two boundary rows **first** allows initiating the respective halo transfers early.
The bulk region can then run while these transfers are in flight.


<div style="text-align: center;">
  <table>
    <tr>
      <td style="text-align: center;">
        <img src="./img/halo-exchange-overlap-0.png" height="320" alt="Halo Exchange Visualization"><br>
        <div>We focus on a single patch with two neighbors (top and bottom).</div>
      </td>
      <td style="text-align: center;">
        <img src="./img/halo-exchange-overlap-1.png" height="320" alt="Halo Exchange Visualization"><br>
        <div>Compute the top row.</div>
      </td>
      <td style="text-align: center;">
        <img src="./img/halo-exchange-overlap-2.png" height="320" alt="Halo Exchange Visualization"><br>
        <div>Push the top row to the top neighbor.</div>
      </td>
      <td style="text-align: center;">
        <img src="./img/halo-exchange-overlap-3.png" height="320" alt="Halo Exchange Visualization"><br>
        <div>Compute the bottom row.</div>
      </td>
      <td style="text-align: center;">
        <img src="./img/halo-exchange-overlap-4.png" height="320" alt="Halo Exchange Visualization"><br>
        <div>Push the bottom row to the bottom neighbor.</div>
      </td>
      <td style="text-align: center;">
        <img src="./img/halo-exchange-overlap-5.png" height="320" alt="Halo Exchange Visualization"><br>
        <div>Compute the remaining bulk part of the patch interior.</div>
      </td>
    </tr>
  </table>
</div>

This adapted update scheme now allows overlapping operations using streams, including hiding the communication overhead behind the bulk computation.

<div style="text-align: center;">
  <table>
    <tr>
      <td style="text-align: center;">
        <img src="./img/halo-exchange-overlap-6.png" height="320" alt="Halo Exchange Visualization"/>
      </td>
      <td style="text-align: center;">
        <img src="./img/halo-exchange-overlap-streams.png" width="960" style="background-color:white" alt="Halo Exchange Stream Visualization"/>
      </td>
    </tr>
  </table>
</div>

### 1. Decompose the Iteration Space

We start by splitting the iteration space of each patch:
* The bottom boundary row (first inner row) produces data for the lower neighbor's top halo
* The top boundary row (last inner row) produces data for the upper neighbor's bottom halo
* The bulk region computes the remaining interior rows

For each of the sub-spaces, a single kernel can be launched.
Note that we use a small, specialized CUDA grid for the boundary kernels as to not waste idle threads.
Due to the way we designed our kernel, merely adapting its index parameters is sufficient.

```c++
// y: [1, 2)
stencil2D<<<ceilingDivide(patch.localNumCellsX - 2, 256), 256>>>(
        patch.localU, patch.localUNew,
        1, patch.localNumCellsX - 1,
        1, 2,
        patch.localNumCellsX);

// y: [localNumCellsY - 2, localNumCellsY - 1)
stencil2D<<<ceilingDivide(patch.localNumCellsX - 2, 256), 256>>>(
        patch.localU, patch.localUNew,
        1, patch.localNumCellsX - 1,
        patch.localNumCellsY - 2, patch.localNumCellsY - 1,
        patch.localNumCellsX);

// y: [2, localNumCellsY - 2)
stencil2D<<<patch.gridSize, patch.blockSize>>>(
        patch.localU, patch.localUNew,
        1, patch.localNumCellsX - 1,
        2, patch.localNumCellsY - 2,
        patch.localNumCellsX);
```


### 2. Add Streams

The version from step 1 runs all kernels in the **default stream** which serializes their execution.
To enable parallel execution, we use three CUDA streams per patch which we manage as part of our `Patch` structure.

```c++
struct Patch {
    // ... as before

    // patch streams
    cudaStream_t topStream;
    cudaStream_t bottomStream;
    cudaStream_t bulkStream;
};
```


As before, we create the streams during the setup phase with
```c++
checkCudaError(cudaStreamCreate(&patch.anyOfTheStreams));
```
and deallocate it at the end of our application with
```c++
checkCudaError(cudaStreamDestroy(patch.anyOfTheStreams));
```
and use them in kernel launches with
```c++
stencil2D<<<grid, block, 0, patch.anyOfTheStreams>>>(...);
```

Important: stream creation (and kernel launches) are specific to the currently active device.
Remember to set it with
```c++
checkCudaError(cudaSetDevice(patchIdx);
```


### 3. Move Data Transfers to Streams

Since we are already using `cudaMemcpyAsync`, the only change necessary here is using the correct stream for each operation.
```c++
cudaMemcpyAsync(...,
        cudaMemcpyDeviceToDevice,
        patch.anyOfTheStreams
);
```


### 4. Fix Order and Synchronization

There are multiple possible implementations for this step, but we follow a more straight-forward approach:

1. Handle the bottom row: enqueue the kernel into the `bottomStream`, then enqueue the asynchronous copy into the same stream.
   Because both operations are enqueued on the same stream, CUDA ensures that the copy starts only after the boundary kernel completed.
2. Follow the same process to handle the top row and its transfer using the `topStream`.
3. Launch the bulk kernel on `bulkStream`.
4. Synchronize with the three streams.
   This can be done individually using `cudaStreamSynchronize(patch.anyOfTheStreams)`, or with a single `cudaDeviceSynchronize()` operation.
   In practice, the former is recommended due to a more fine-grained control, avoiding side-effects and higher verbosity.
   We will, however, go with the latter since it's more straight-forward to use and can be more robust.

### 5. Additional Considerations

Because kernel launches and device-to-device copies are now asynchronous, the GPU **can** progress the halo copies while executing the bulk kernel.
This is not a guarantee however - profiling is recommended as usual.

Be careful to avoid race conditions:
* Write only to remote data elements that are not read in the same time step (`uNew`, and outside of the region touched by the patches kernels).
* Synchronize before swapping the pointers to make sure that all patches are synchronized.
* Synchronize between boundary kernels and data transfers of produced data - we do this by using a single stream for both operations.

## Exercise

Choose one of the difficulty levels below to tailor the exercise to your preferences.
Each level provides a different starting point implementation to be copied into [stencil-2d.cu](../src/09-overlap/stencil-2d.cu).
Use it for your implementation and follow the steps outlined above.

Below the difficulty level descriptions, there are cells for compiling, executing and profiling the your solution.

### Level Hard

Create and empty file [stencil-2d.cu](../src/09-overlap/stencil-2d.cu) and copy one of the previous code versions into it (your work or a solution).

In [ ]:
!touch ../src/09-overlap/stencil-2d.cu

### Level Medium

[stencil-2d-medium.cu](../src/09-overlap/stencil-2d-medium.cu) contains a partial solution with TODOs ranging from straight-forward to complex.
Copy the provided code into the working file [stencil-2d.cu](../src/09-overlap/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
%%bash
if [ -e ../src/09-overlap/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/09-overlap/stencil-2d-medium.cu ../src/09-overlap/stencil-2d.cu
fi

### Level Easier

[stencil-2d-easier.cu](../src/09-overlap/stencil-2d-easier.cu) contains a partial solution with TODOs.
The solution is further progressed than the level medium version, and the complexity of the TODOs is limited.
Copy the provided code into the working file [stencil-2d.cu](../src/09-overlap/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
%%bash
if [ -e ../src/09-overlap/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/09-overlap/stencil-2d-easier.cu ../src/09-overlap/stencil-2d.cu
fi

### Possible Solution

[stencil-2d-solution.cu](../src/09-overlap/stencil-2d-solution.cu) contains a possible solution for this exercise.
Copy it into the working file [stencil-2d.cu](../src/09-overlap/stencil-2d.cu) with the cell below.

In [ ]:
%%bash
if [ -e ../src/09-overlap/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/09-overlap/stencil-2d-solution.cu ../src/09-overlap/stencil-2d.cu
fi

### Compilation, Execution and Profiling

The new code version is available in [09-overlap/stencil-2d.cu](../src/09-overlap/stencil-2d.cu) (after creating it with one of the commands above).
It can be compiled and executed with the following cells.

In [ ]:
!nvcc -O3 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -o ../build/09-overlap ../src/09-overlap/stencil-2d.cu

The next cell produces output for a small grid.
Visualize the output using the [visualize](./99-visualize.ipynb) notebook after executing the application.

In [ ]:
!../build/09-overlap 256 64 2 2000 100

The next cell produces no output and runs a larger grid.
Use it for performance evaluation.

In [ ]:
!../build/09-overlap $((32 * 1024)) 256 2 16 0

Instead of running on the local **single A40** GPU, we can also submit a batch job running on **multiple A100** GPUs.
Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between one and eight GPUs.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/09-overlap.out --error=../output/09-overlap.err \
    --wrap="../build/09-overlap $((32 * 1024)) 256 2 8 0"

cat ../output/09-overlap.out

The next cell performs profiling with Nsight Systems by submitting a batch job.
Feel free to tune the number of GPUs in the `--gres=gpu:a100:NGPU` to anything between one and eight GPUs.

The profile is then available at [profiles/09-overlap.nsys-rep](../profiles/09-overlap.nsys-rep) and can be downloaded by **shift + right-clicking** the link, by clicking the link with the **middle mouse button**, or using the JupyterHub file tree.

After downloading it, open it up locally to visualize the run-time behavior of your application.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/09-overlap-nsys.out --error=../output/09-overlap-nsys.err \
    --wrap="nsys profile --stats=true --force-overwrite=true \
        -o ../profiles/09-overlap \
        ../build/09-overlap $((32 * 1024)) 256 2 8 0"

cat ../output/09-overlap-nsys.out

## Next Step

The next step is diving into an MPI introduction in [10](./10-mpi-hw.ipynb).